## **Crawling Data Olahraga dan Otomotif**

### **Apa itu Crawling?**

Proses crawling adalah penggunaan program komputer untuk secara otomatis menjelajahi dan mengumpulkan data dari berbagai halaman web di internet. Proses ini dilakukan oleh bot yang disebut web crawlers atau spiders, yang akan menelusuri situs web, mengakses halaman-halaman, dan mengunduh atau mengekstraksi informasi yang dibutuhkan sebelum disimpan atau diindeks dalam database.

### **Berikut adalah beberapa fungsi dari crawling:**

1. **Data Collection**: Crawling digunakan untuk mengumpulkan data dari berbagai sumber, seperti situs web, forum, dan lain-lain. Data yang dikumpulkan dapat berupa teks, gambar, video, dan lain-lain.
2. **Indexing**: Crawling digunakan untuk membuat indeks dari data yang dikumpulkan. Indeks ini dapat digunakan untuk mencari data yang relevan dengan kata kunci atau query.
3. **Search Engine Optimization (SEO)**: Crawling digunakan oleh search engine untuk mengindeks situs web dan membuat daftar hasil pencarian yang relevan.
4. **Market Research**: Crawling digunakan untuk mengumpulkan data tentang produk, harga, dan lain-lain dari berbagai sumber, seperti e-commerce, forum, dan lain-lain.
5. **Competitor Analysis**: Crawling digunakan untuk mengumpulkan data tentang kompetitor, seperti produk, harga, dan lain-lain, untuk analisis dan strategi bisnis.
6. **Data Mining**: Crawling digunakan untuk mengumpulkan data yang besar dan kompleks, seperti data transaksi, data pengguna, dan lain-lain, untuk analisis dan penelitian.
7. **Web Scraping**: Crawling digunakan untuk mengumpulkan data dari situs web yang tidak memiliki API atau tidak memungkinkan akses langsung ke data.
8. **Social Media Monitoring**: Crawling digunakan untuk mengumpulkan data dari berbagai platform sosial media, seperti Twitter, Facebook, dan lain-lain, untuk analisis dan monitoring.
9. **Email Harvesting**: Crawling digunakan untuk mengumpulkan alamat email dari berbagai sumber, seperti situs web, forum, dan lain-lain.
10. **Web Archiving**: Crawling digunakan untuk mengumpulkan data dari situs web yang tidak lagi aktif atau tidak dapat diakses, untuk tujuan archiving dan preservasi.

Dalam keseluruhan, crawling adalah proses yang sangat berguna untuk mengumpulkan data dari berbagai sumber, dan dapat digunakan untuk berbagai tujuan, seperti analisis, penelitian, dan bisnis.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/PPW

/content/drive/MyDrive/PPW


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# Fungsi untuk membersihkan konten dari elemen-elemen yang tidak diinginkan
def clean_content(content_element):
    if content_element:
        # Hapus elemen yang berisi daftar isi
        for daftar_isi in ["collapsible"]:
            unwanted = content_element.find("div", id=daftar_isi)
            if unwanted:
                unwanted.decompose()

        # Hapus elemen yang berisi tag
        for tag_class in ["aevp", "detail__body-tag mgt-16"]:
            unwanted = content_element.find_all("div", class_=tag_class)
            for el in unwanted:
                el.decompose()

        # Hapus elemen yang berisi link sisipan
        link_sisip = content_element.find_all("table", class_="linksisip")
        for table in link_sisip:
            table.decompose()

        # Hapus elemen paragraf dan span dengan class 'para_caption'
        unwanted_paragraphs = content_element.find_all(["p", "span"], class_="para_caption")
        for para in unwanted_paragraphs:
            para.decompose()

        # Kembalikan teks yang tersisa
        return content_element.get_text(separator=' ', strip=True).strip()

    return "Content Not Found"

In [ ]:
# Fungsi untuk mengambil data dari halaman web Detik.com
def get_data(url, kategori, min_articles_per_category):
    try:
        response = requests.get(url)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return

    soup = BeautifulSoup(response.content, "html.parser")
    articles = soup.find_all("article", class_="list-content__item")

    for article in articles:
        if len([k for k in kategori_list if k == kategori]) >= min_articles_per_category:
            return  # Menghentikan proses jika jumlah artikel sudah mencapai minimum yang diinginkan

        try:
            link = article.find("a")["href"]
            article_response = requests.get(link)
            article_response.raise_for_status()
        except (requests.exceptions.RequestException, TypeError) as e:
            print(f"Request for article failed: {e}")
            continue

        article_soup = BeautifulSoup(article_response.content, "html.parser")
        title_element = article_soup.find("h1", class_="detail__title")
        title = title_element.text.strip() if title_element else "Title Not Found"
        date_element = article_soup.find("div", class_="detail__date")
        date = date_element.text.strip() if date_element else "Date Not Found"
        content_element = article_soup.find("div", class_="detail__body-text")
        content = content_element.text.strip() if content_element else "Content Not Found"

        # Bersihkan konten menggunakan fungsi clean_content
        content = clean_content(content_element)


        judul.append(title)
        tanggal.append(date)
        isi.append(content)
        # url_list.append(link)
        kategori_list.append(kategori)

        if len(judul) <= 10:
          print(title)
        time.sleep(1)  # Menambahkan jeda waktu 1 detik antara permintaan artikel

# Membuat list url dan kategori yang akan di-crawl
base_urls = [
    "https://sport.detik.com/indeks",
    "https://oto.detik.com/indeks",
]
categories = [
    "Olahraga",
    "Otomotif",
]

# Inisialisasi list untuk menyimpan data
judul = []
tanggal = []
isi = []
kategori_list = []

# Batas minimal artikel per kategori
min_articles_per_category = 50

# Melakukan iterasi untuk setiap url dan kategori
for base_url, category in zip(base_urls, categories):
    page = 1
    while len([k for k in kategori_list if k == category]) < min_articles_per_category:
        url = f"{base_url}/{page}"
        get_data(url, category, min_articles_per_category)
        time.sleep(2)  # Menambahkan jeda waktu 2 detik antara permintaan halaman
        page += 1

# Membuat dataframe dari list data
df = pd.DataFrame({"judul": judul, "tanggal": tanggal, "isi": isi, "kategori": kategori_list})

# Menyimpan dataframe ke file csv
df.to_csv("data_berita.csv", index=False)

# Menampilkan dataframe
# print(df)

Apriyani Rahayu Sudah Berlatih Lagi, Ingin Comeback Tahun Depan
Hasil Korea Masters 2024: Putri KW Lanjut ke 16 Besar
Fajar/Rian di Japan dan China Masters: Inginnya Juara
Hasil Korea Masters 2024: Komang Ayu Tersingkir
Korea Masters 2024 Hari Kedua: Tiga Wakil RI Berjuang ke 16 Besar
Korea Masters 2024: Fikri/Daniel Puji Lawan Usai Lolos 16 Besar
Catalunya Resmi Gantikan Valencia Jadi Seri Terakhir MotoGP 2024
Korea Masters 2024: Fikri/Daniel Singkirkan Pasangan Malaysia
Hitung-hitungan Jorge Martin Bisa Juara MotoGP 2024
Ranking BWF: Lanny/Fadia Melesat 38 Anak Tangga Dunia


In [ ]:
df=pd.read_csv("data_berita.csv")
df

,judul,tanggal,isi,kategori
0,"Apriyani Rahayu Sudah Berlatih Lagi, Ingin Com...","Rabu, 06 Nov 2024 15:25 WIB",Jakarta - Atlet bulutangkis Indonesia Apriyani...,Olahraga
1,Hasil Korea Masters 2024: Putri KW Lanjut ke 1...,"Rabu, 06 Nov 2024 14:36 WIB",Jakarta - Putri Kusuma Wardani hadapi wakil Ta...,Olahraga
2,Fajar/Rian di Japan dan China Masters: Inginny...,"Rabu, 06 Nov 2024 14:13 WIB",Jakarta - Fajar Alfian/Muhammad Rian Ardianto ...,Olahraga
3,Hasil Korea Masters 2024: Komang Ayu Tersingkir,"Rabu, 06 Nov 2024 12:45 WIB",Jakarta - Komang Ayu Cahya Dewi hadapi Liang T...,Olahraga
4,Korea Masters 2024 Hari Kedua: Tiga Wakil RI B...,"Rabu, 06 Nov 2024 10:55 WIB",Jakarta - Turnamen Korea Masters 2024 memasuki...,Olahraga
...,...,...,...,...
95,Inikah Motor Baru Honda yang Meluncur di Indon...,"Selasa, 05 Nov 2024 07:14 WIB",Jakarta - PT Astra Honda Motor (AHM) akan melu...,Otomotif
96,Kenapa Harus Ganti Aki Meski Mobil Masih Kuat ...,"Senin, 04 Nov 2024 20:25 WIB",Jakarta - Ketika mobil masih bisa distater ban...,Otomotif
97,"Biaya Bikin SIM A: Syarat, Cara Buat Baru, dan...","Senin, 04 Nov 2024 19:38 WIB",Jakarta - Pengemudi mobil diwajibkan memiliki ...,Otomotif
98,"Penjualan Mobil Turun, Kok Permintaan Motor Ma...","Senin, 04 Nov 2024 19:06 WIB","Jakarta - Berbeda dengan mobil, penjualan moto...",Otomotif
